# 1 — Extended Lab: Univariate Time Series Modeling (AR → MA → ARMA → ARIMA → SARIMA → Backtesting)

**Prerequisite reading:** `00_Background_and_Theory.ipynb`.
**Where to do the work:** `02_Skeleton_Practice.ipynb` (stubs) → check yourself against `05_Solutions.ipynb` only after attempting each part.
**Dataset for Parts D–F:** `COCO_COLA.csv` (daily Coca-Cola stock data, 1962-01-19 to 2021-12-19; columns `Date, Open, High, Low, Close, Adj Close, Volume`).

This lab extends the chapter's worked examples with additional orders, an additional real dataset, and explicit hold-out backtesting the source material only partially demonstrates. Each part ends with **questions you must answer in a markdown cell**, not just code to run.

---
## Part A — AR(p): identification and estimation from a simulated process

1. Simulate an **AR(3)** process of your own choosing with `statsmodels.tsa.ArmaProcess`, using AR coefficients such that the process is stationary (verify with `.arroots` / `.isstationary`). Generate `n=300` samples.
2. Plot the realization, ACF, and PACF (`plot_acf`, `plot_pacf`, 40 lags, `alpha=0.05`).
3. From the PACF alone, state what order $p$ you would select and why (cite the specific lag(s) that are/aren't significant).
4. Run the Augmented Dickey-Fuller test with a `maxlag` justified by your ACF plot. Report the p-value and your conclusion about a trend.
5. Use `arma_order_select_ic` with `max_ar=6, max_ma=0` to get AIC and BIC order recommendations. Do they agree with each other and with your visual read?
6. Fit an `ARIMA(p,0,0)` model with `enforce_stationarity=True` at **both** your visually-selected order and the AIC-selected order (fit both if they differ). Compare the two fits on: coefficient significance, AIC, and log-likelihood.
7. **Q1:** Which of your two candidate orders would you ship, and why? Justify using both statistical evidence (coefficient p-values) and the overfitting risk discussed in the Background notebook.

## Part B — MA(q) and ARMA(p,q): identification and invertibility

1. Simulate an **MA(2)** process with complex-conjugate roots (pick $\theta_1,\theta_2$ so the roots are complex). Verify invertibility numerically and by hand (compute the root magnitude with the quadratic formula and compare to the `.maroots` output).
2. Plot ACF/PACF and confirm the "MA cuts off in ACF, decays in PACF" pattern from the Background notebook.
3. Simulate a stationary, invertible **ARMA(2,2)** process. Run the full workflow: visual inspection → ADF test → `arma_order_select_ic` (try `max_ar=4, max_ma=3`) → fit the AIC-selected order → fit the BIC-selected order if it differs.
4. Read the `.summary()` diagnostics (Ljung-Box, Jarque-Bera, Heteroskedasticity, skew/kurtosis) for your final chosen model and interpret each one in a sentence.
5. Do a 5-step-ahead **test forecast**: fit on all but the last 5 points, forecast those 5, and report the Average Squared Error against the true held-out values.
6. **Q2:** The book shows a case where AIC selects a higher order (ARMA(4,1)) with insignificant coefficients, while BIC's more parsimonious pick (ARMA(2,1)) is all-significant. Did you encounter something similar? If your AIC and BIC orders agree, explain the trade-off anyway — what would you look for if they hadn't agreed?

## Part C — ARIMA and Seasonal (differencing-only) ARIMA on real data

1. Load `y_macro_economic = sktime.datasets.load_macroeconomic()` and `y_airline = sktime.datasets.load_airline()`.
2. For the GDP series (`realgdp`): plot the series and ACF; run the ADF test; take a first difference; re-run the ADF test on the differenced series; state the resulting $d$.
3. Use `pm.auto_arima` on the **differenced** GDP series (`seasonal=False`) to find $(p,q)$, then state the full $(p,d,q)$ order for the *original* series.
4. Split GDP 90/10 (train/test) with `pmdarima.model_selection.train_test_split`, fit `pm.auto_arima` on the train set, forecast the test horizon, and plot prediction vs. actual with confidence interval. Report RMSE on the test set.
5. Repeat steps 2–4 for the airline-passenger series, but this time identify the **seasonal period** from the ACF, take a **seasonal difference** at that lag, check whether an additional first difference is still needed (ADF test), and use `pm.auto_arima(..., seasonal=True, m=<period>)` for the automatic fit.
6. **Q3:** Compare the test-set forecast quality (visually and by RMSE) between the GDP model and the airline model. Which model tracks its held-out actuals more closely, and what property of the underlying series (discussed in the Background notebook) explains the difference?

## Part D — Real-world data exploration: Coca-Cola stock

1. Load `COCO_COLA.csv`, parse `Date` as the index, and restrict to **`Open` prices from 2016-01-01 onward** (matches the book's worked example).
2. Reproduce the **resampling** comparison: plot the daily series alongside 7-day, monthly (`'M'`), and yearly (`'Y'`) resampled means, in a 4-panel figure. In one sentence, describe the trade-off you observe as the frequency decreases.
3. Create a `price_lag_1` feature with `.shift(1)`, first with the default `NaN` fill, then with `fill_value=data['Open'].mean()`. **Q4:** Under what circumstances would filling the first lag with the series mean silently bias a model trained on this feature?

## Part E — Baselines that any real model must beat

Using the **monthly-resampled** Coca-Cola `Open` series from 2016 onward, with the same 24-month train/test convention as the book (`[0:-24]` / `[-24:]`):

1. **Optimized persistence forecasting** — sweep $p = 1,\dots,24$ (forecast = value from $p$ months ago), compute RMSE for each, plot RMSE vs. $p$, and identify the best $p$.
2. **Rolling-window forecasting** — sweep window size $w = 1,\dots,24$ (forecast = mean of the last $w$ months), compute RMSE for each, plot RMSE vs. $w$, and identify the best $w$.
3. Plot the best persistence forecast and the best rolling forecast against the actual test values on the same axes.
4. **Q5:** Which of the two naive baselines performs better on this series, by best-case RMSE? Does either look visually trustworthy around the 2020 dip? What does that tell you about naive baselines under a structural break?

## Part F — Put it together: does ARIMA beat the naive baselines?

1. On the **same** monthly Coca-Cola train/test split used in Part E, fit `pm.auto_arima` (try both `seasonal=False` and `seasonal=True, m=12`) on the training portion.
2. Forecast the 24-month test horizon and compute RMSE.
3. **Q6 (capstone question):** Compare your ARIMA/SARIMA RMSE to the best persistence RMSE and best rolling-window RMSE from Part E. Did the more sophisticated model actually win? Write a short paragraph (5-8 sentences) that: (a) states the winner, (b) proposes a reason grounded in the Background notebook's "Limitations" section for *why* that model won or didn't, and (c) states what you would try next if this were a real forecasting task (e.g., exogenous variables, a volatility model, a different horizon).

## Part G (open-ended, optional) — Apply the workflow to a series of your choice

Pick any univariate time series you have access to (a public dataset, a personal dataset, or another column from `COCO_COLA.csv` such as `Volume` or `Close`). Using `04_Reusable_Project_Template.ipynb` as your starting point, run the full identification → fit → diagnose → backtest workflow and write up your findings in 1 page or less.

---
### Deliverable checklist
- [ ] All code cells run top-to-bottom without errors
- [ ] Every numbered question (Q1–Q6) answered in its own markdown cell, with evidence (numbers/plots), not just an opinion
- [ ] At least one instance where you explicitly reject a higher-AIC/lower-error model in favor of a more defensible one, with your reasoning stated
